# 08 — Similarity Search: Find Similar Past Work Orders

Goal: Given a new work order description, find the N most similar past cases.

This is the most practically valuable feature:
> 'A technician describing a new problem can instantly see what fixed similar past failures,
> reducing diagnosis time.'

Pipeline:
1. Embed all work orders with sentence-transformers ('all-MiniLM-L6-v2')
2. Build cosine similarity index
3. Query: new text → embedding → nearest neighbors
4. Save embeddings index to models/ for API use

In [ ]:
import pandas as pd
import numpy as np
import sys; sys.path.insert(0, '..')
from src.nlp_pipeline import embed, save_embeddings, cosine_similarity_search

df = pd.read_csv('../data/work_orders.csv')
print(f'Embedding {len(df):,} work orders...')

In [ ]:
# Embed corpus (~2 min on CPU for 3,000 texts)
embeddings = embed(df.text.tolist(), model_name='all-MiniLM-L6-v2', batch_size=64)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Save to models/ for API use
save_embeddings(embeddings, df.text.tolist(), out_dir='../models')

In [ ]:
# Test query
query = 'Pump P-104 tripped on vibration alarm. Found worn bearings after opening. Root cause: contaminated lube oil.'
q_emb = embed([query], show_progress=False)[0]
hits = cosine_similarity_search(q_emb, embeddings, top_k=5)

print(f'Query: {query}\n')
for rank, (idx, score) in enumerate(hits, 1):
    row = df.iloc[idx]
    print(f'[{rank}] Score: {score:.3f} | Category: {row.failure_category}')
    print(f'    {row.text[:200]}...')
    print()

In [ ]:
# Evaluate retrieval quality: for each test record, does top-1 match the same category?
np.random.seed(42)
test_idx = np.random.choice(len(df), 200, replace=False)
correct = 0
for i in test_idx:
    q = embeddings[i]
    # exclude self
    mask = np.ones(len(embeddings), dtype=bool)
    mask[i] = False
    hits = cosine_similarity_search(q, embeddings[mask], top_k=1)
    best_idx = hits[0][0]
    # adjust index for mask
    if best_idx >= i: best_idx += 1
    if df.iloc[best_idx].failure_category == df.iloc[i].failure_category:
        correct += 1
print(f'Same-category retrieval accuracy @ top-1: {correct/len(test_idx):.1%}')
# Expected: 85–95% — same failure category in top-1 similar work order